# kinfast quickstart

Load a real robot, solve 10,000 IK problems in one batch, and compile the robot
down to microsecond forward kinematics. Runs on CPU; on a Colab GPU runtime
(Runtime > Change runtime type > T4 GPU) the batched numbers get much bigger.

Repo: https://github.com/VihanAggarwal/kinfast

In [ ]:
try:
    import kinfast
except ImportError:
    %pip install -q git+https://github.com/VihanAggarwal/kinfast
    import kinfast
print("kinfast", kinfast.__version__)

In [ ]:
# grab a real robot: the Franka Panda URDF from bullet3
import os, urllib.request
URDF = "panda.urdf"
if not os.path.exists(URDF):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bulletphysics/bullet3/master/"
        "examples/pybullet/gym/pybullet_data/franka_panda/panda.urdf", URDF)
robot = kinfast.load(URDF)
print(robot.dof, "dof:", robot.joint_names)

## 10,000 IK problems in one batch

Targets come from the robot's own FK, so they are all reachable. Everything is
differentiable end to end.

In [ ]:
import time, torch
device = "cuda" if torch.cuda.is_available() else "cpu"
robot = robot.to(device)
print("device:", device)

n = 10_000
targets = robot.fk(robot.random_configs(n))
if device == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
q_sol, info = robot.ik(targets, iters=100, pos_only=True, restarts=4)
if device == "cuda":
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
err = (robot.fk(q_sol)[:, :3, 3] - targets[:, :3, 3]).norm(dim=-1)
print(f"{n:,} IK targets ({n*4:,} seeds) in {dt:.1f}s "
      f"({n*4/dt:,.0f} solves/s), {(err < 5e-2).float().mean()*100:.1f}% within 5cm")

## The compiler: microsecond single-query FK

`robot.compile()` generates straight-line code specialized to this exact robot.
This is the path a control loop or planner wants.

In [ ]:
fast = kinfast.load(URDF).compile()
ql = [0.0] * fast.dof
for _ in range(100):
    fast._raw(ql)                      # warmup
t0 = time.perf_counter()
N = 5000
for _ in range(N):
    fast._raw(ql)
us = (time.perf_counter() - t0) / N * 1e6
print(f"single-query FK: {us:.1f} us per call ({1e6/us:,.0f} Hz)")
print("\ngenerated source, first lines:")
print("\n".join(fast.source.splitlines()[:12]))

## MJCF works too

Same loader, same API. This is the SO-ARM100 straight from the MuJoCo
Menagerie.

In [ ]:
MJCF = "so_arm100.xml"
if not os.path.exists(MJCF):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/"
        "main/trs_so_arm100/so_arm100.xml", MJCF)
arm = kinfast.load(MJCF)
targets = arm.fk(arm.random_configs(256))
q_sol, _ = arm.ik(targets, iters=100, pos_only=True, restarts=8)
err = (arm.fk(q_sol)[:, :3, 3] - targets[:, :3, 3]).norm(dim=-1)
print(f"SO-ARM100 (MJCF): {arm.dof} dof, "
      f"{(err < 5e-2).float().mean()*100:.1f}% of 256 IK targets within 5cm")

## Where the robot can reach


In [ ]:
from kinfast import analysis
import matplotlib.pyplot as plt
ws = analysis.workspace(arm.chain, arm.link_id(arm.ee_link), n=20000)
pts = ws["points"].cpu().numpy()
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts[:, 0], pts[:, 2], s=1, alpha=0.15)
ax.set_xlabel("x [m]"); ax.set_ylabel("z [m]"); ax.set_aspect("equal")
ax.set_title(f"SO-ARM100 workspace, max reach {ws['max_reach']:.2f} m")
plt.show()

## More

- dynamics: `robot.mass_matrix(q)`, `robot.inverse_dynamics(q, qd, qdd)`
- trajectories: `robot.point_to_point(q_a, q_b)` (limit-safe trapezoid)
- collision-aware IK: `kinfast.collision.collision_aware_ik`
- SO-101 walkthrough: `docs/SO101_TUTORIAL.md` in the repo

If you have a URDF or MJCF that does not load, please open an issue with the
file: https://github.com/VihanAggarwal/kinfast/issues